# 18 — Merge & Join

Combining DataFrames is the most critical operation in data engineering.
This notebook covers all join types, merge strategies, index joins, concat,
and anti-joins — with the exact interview questions around each.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

# --- Core tables ---
employees = pd.DataFrame({
    'emp_id':      [1001, 1002, 1003, 1004, 1005, 1006, 1007],
    'name':        ['Alice', 'Bob', 'Carol', 'Dave', 'Eve', 'Frank', 'Grace'],
    'dept_id':     [101, 102, 101, 103, 102, 104, 101],
    'salary':      [90000, 85000, 95000, 70000, 80000, 110000, 88000],
    'manager_id':  [None, 1001, 1001, 1002, 1002, None, 1001],
})

departments = pd.DataFrame({
    'dept_id':   [101, 102, 103, 105],  # 104 missing!, 105 has no employees
    'dept_name': ['Engineering', 'Sales', 'HR', 'Marketing'],
    'location':  ['Bangalore', 'Mumbai', 'Delhi', 'Pune'],
})

salaries = pd.DataFrame({
    'emp_id':   [1001, 1002, 1003, 1004, 1008],  # 1008 doesn't exist
    'bonus':    [15000, 12000, 18000, 8000, 5000],
    'year':     [2023, 2023, 2023, 2023, 2023]
})

print("employees:")
print(employees)
print("\ndepartments:")
print(departments)

## 1. The Four Join Types

In [ ]:
# INNER JOIN — only matching rows
inner = pd.merge(employees, departments, on='dept_id', how='inner')
print(f"INNER: {len(inner)} rows")
print(inner)

In [ ]:
# LEFT JOIN — all employees, dept_name=NaN if no match
left = pd.merge(employees, departments, on='dept_id', how='left')
print(f"LEFT: {len(left)} rows")
print(left)

In [ ]:
# RIGHT JOIN — all departments, even if no employees
right = pd.merge(employees, departments, on='dept_id', how='right')
print(f"RIGHT: {len(right)} rows")
print(right)

In [ ]:
# OUTER JOIN — all rows from both sides, NaN where no match
outer = pd.merge(employees, departments, on='dept_id', how='outer')
print(f"OUTER: {len(outer)} rows")
print(outer)

## 2. Join on Different Column Names

In [ ]:
# Left table has 'emp_id', right has 'emp_id' — same name (trivial)
# But what if column names differ?

orders = pd.DataFrame({
    'order_id': [1, 2, 3, 4],
    'client_id': [101, 102, 101, 103]
})
clients = pd.DataFrame({
    'customer_id': [101, 102, 104],  # different column name
    'client_name': ['Acme', 'Beta', 'Gamma']
})

# Use left_on / right_on when column names differ
merged = pd.merge(orders, clients, left_on='client_id', right_on='customer_id', how='left')
print(merged)

## 3. Merging on Multiple Keys

In [ ]:
# Multi-key merge: must match on BOTH keys
prices = pd.DataFrame({
    'product': ['Laptop', 'Phone', 'Laptop', 'Phone'],
    'region':  ['North', 'North', 'South', 'South'],
    'price':   [80000, 30000, 85000, 32000]
})
orders2 = pd.DataFrame({
    'order_id': [1, 2, 3, 4, 5],
    'product':  ['Laptop', 'Phone', 'Laptop', 'Phone', 'Tablet'],
    'region':   ['North', 'South', 'South', 'North', 'North'],
    'qty':      [2, 5, 1, 3, 4]
})

enriched = pd.merge(orders2, prices, on=['product', 'region'], how='left')
enriched['revenue'] = enriched['qty'] * enriched['price']
print(enriched)

## 4. Handling Duplicate Column Names — suffixes

In [ ]:
# When both tables have a column with the same name (other than the key)
emp_dept = pd.merge(
    employees.assign(salary_rank=employees['salary'].rank(ascending=False)),
    departments.assign(dept_salary_budget=[200000, 350000, 100000, 80000]),
    on='dept_id',
    how='left',
    suffixes=('_emp', '_dept')  # disambiguate same-name columns
)
print(emp_dept.columns.tolist())

## 5. validate= — Ensure Join Cardinality

In [ ]:
# validate catches unexpected duplicates in join keys
# '1:1'  — both sides unique
# 'm:1'  — left may have duplicates, right is unique
# '1:m'  — left unique, right may have duplicates
# 'm:m'  — no constraints

try:
    result = pd.merge(
        employees, departments,
        on='dept_id', how='left',
        validate='m:1'  # many employees to 1 department (expected)
    )
    print(f"m:1 join validated: {result.shape}")
except Exception as e:
    print(f"Validation failed: {e}")

try:
    # This SHOULD fail — employees has repeated dept_ids (many per dept)
    pd.merge(employees, departments, on='dept_id', validate='1:1')
except Exception as e:
    print(f"\n1:1 validation correctly failed: {type(e).__name__}")

## 6. indicator= — Diagnose Join Results

In [ ]:
# indicator=True adds '_merge' column: 'left_only', 'right_only', 'both'
outer_with_indicator = pd.merge(
    employees, departments,
    on='dept_id', how='outer',
    indicator=True
)
print(outer_with_indicator[['name', 'dept_id', 'dept_name', '_merge']])
print("\nMatch distribution:")
print(outer_with_indicator['_merge'].value_counts())

## 7. Anti-Join — Rows NOT in Other Table

In [ ]:
# Left anti-join: employees with no department (Frank — dept_id=104 missing in depts)
anti_left = (
    pd.merge(employees, departments, on='dept_id', how='left', indicator=True)
    .query('_merge == "left_only"')
    .drop(columns='_merge')
)
print("Employees with unknown department:")
print(anti_left[['emp_id', 'name', 'dept_id', 'salary']])

In [ ]:
# Right anti-join: departments with no employees
anti_right = (
    pd.merge(employees, departments, on='dept_id', how='right', indicator=True)
    .query('_merge == "right_only"')
    .drop(columns='_merge')
)
print("Departments with no employees:")
print(anti_right[['dept_id', 'dept_name', 'location']])

## 8. Self-Join — Manager Lookup

In [ ]:
# Self-join to get manager name
emp_with_manager = pd.merge(
    employees,
    employees[['emp_id', 'name']].rename(columns={'emp_id': 'manager_id', 'name': 'manager_name'}),
    on='manager_id',
    how='left'
)
print(emp_with_manager[['emp_id', 'name', 'manager_id', 'manager_name']])

## 9. concat() — Stacking DataFrames

In [ ]:
# Stack vertically (axis=0) — must have same columns (or get NaN)
emp_2022 = pd.DataFrame({'emp_id': [2001, 2002], 'name': ['Henry', 'Irene'], 'salary': [75000, 80000]})
emp_2023 = pd.DataFrame({'emp_id': [3001, 3002], 'name': ['James', 'Karen'], 'salary': [82000, 88000]})

all_emp = pd.concat([emp_2022, emp_2023], ignore_index=True)
print(all_emp)

In [ ]:
# concat with keys — creates a MultiIndex to track source
all_emp_labeled = pd.concat(
    [emp_2022, emp_2023],
    keys=['2022', '2023'],
    names=['year', 'original_idx']
)
print(all_emp_labeled)
print()
print(all_emp_labeled.loc['2023'])

In [ ]:
# Horizontal concat (axis=1)
emp_info  = pd.DataFrame({'emp_id': [1001, 1002, 1003], 'name': ['Alice', 'Bob', 'Carol']})
emp_stats = pd.DataFrame({'salary': [90000, 85000, 95000], 'rating': [4.5, 3.8, 4.9]})

combined = pd.concat([emp_info, emp_stats], axis=1)
print(combined)

In [ ]:
# concat vs merge — key difference
# concat: positional (no join key) — just stack or place side-by-side
# merge:  relational (join on key) — like SQL JOIN

print("concat: stacks data positionally")
print("merge:  joins on a key column (like SQL)")

## 10. join() — Index-Based Join

In [ ]:
# df.join() is a shortcut for pd.merge(left_index=True, right_index=True)
emp_indexed  = employees.set_index('emp_id')
sal_indexed  = salaries.set_index('emp_id')[['bonus']]

result = emp_indexed.join(sal_indexed, how='left')
print(result)

## 11. Cross Join — All Combinations

In [ ]:
# Cross join: every combination of rows from both tables
products = pd.DataFrame({'product': ['Laptop', 'Phone', 'Tablet'], 'price': [80000, 30000, 20000]})
regions  = pd.DataFrame({'region': ['North', 'South', 'East', 'West']})

cross = pd.merge(products, regions, how='cross')
print(f"Cross join: {len(cross)} rows ({len(products)} × {len(regions)})")
print(cross)

## 12. Merge Performance Tips

| Tip | Reason |
|-----|--------|
| Use `int` or `category` keys | Avoids slow string comparison |
| Sort before merge | Can enable sorted merge algorithms |
| Use `validate=` in prod | Catches unexpected fanout early |
| Use `indicator=True` to debug | See which rows matched |
| Avoid merging on float keys | Floating point equality is unreliable |
| Use `merge_asof` for range joins | Nearest-key matching (not equality) |

In [ ]:
# merge_asof — join on nearest key (sorted)
# Use case: events matched to the most recent reference event
trades = pd.DataFrame({
    'time':  pd.to_datetime(['09:01', '09:03', '09:05', '09:07', '09:10'], format='%H:%M'),
    'price': [100.5, 101.2, 100.8, 102.0, 101.5]
})
quotes = pd.DataFrame({
    'time': pd.to_datetime(['09:00', '09:04', '09:08'], format='%H:%M'),
    'bid':  [100.0, 100.9, 101.8]
})

# Each trade gets the most recent quote before it
result = pd.merge_asof(trades, quotes, on='time', direction='backward')
print(result)